[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/solutions_02_01_exercise_guided.ipynb)

In [ ]:
# --- Course setup (uncomment when running on Colab) ---
#!git clone https://github.com/tunnel-ai/way.git
#import sys; sys.path.insert(0, "/content/way/src")

# Module 2 — Linear Regression (Guided Exercise — Solution)

**Solution to** `02_01_exercise_guided.ipynb`.

**Target:** `log1p(transaction_amount)` (same as the main notebook).

All TODOs filled in, plus answers to the check-in questions at the end.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 1955
np.random.seed(RANDOM_STATE)

In [ ]:
from core.generators.transaction_risk_dgp import generate_transaction_risk_dataset

df = generate_transaction_risk_dataset(seed=RANDOM_STATE)

print(df.shape)
df.head()

## 1) Define X / y and split

In [ ]:
TARGET = "transaction_amount"

DROP_COLS = [
    TARGET,
    # Leakage / label fields
    "transaction_loss_amount",
    "is_fraud",
    "chargeback_flag",
    "manual_review_score",
    "fraud_probability_latent",
    # Pure IDs
    "transaction_id",
    "account_id",
    # Redundant / high-cardinality text
    "merchant_description",
    "merchant_name",
]

X = df.drop(columns=DROP_COLS).copy()
X["merchant_id"] = X["merchant_id"].astype(str)

y = np.log1p(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Test:", X_test.shape)

## 2) Preprocessing pipeline (provided)

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]
low_card_cols = [c for c in categorical_cols if c != "merchant_id"]
high_card_cols = ["merchant_id"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
low_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
high_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat_low", low_card_transformer, low_card_cols),
        ("cat_high", high_card_transformer, high_card_cols),
    ],
    remainder="drop",
)

## 3) Fit OLS and evaluate

In [ ]:
def regression_report(y_true, y_pred, label="model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return pd.Series({"MAE_log": mae, "RMSE_log": rmse, "R2": r2}, name=label)

all_results = []

ols_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression()),
])

ols_model.fit(X_train, y_train)
y_pred = ols_model.predict(X_test)

all_results.append(regression_report(y_test, y_pred, "OLS"))
pd.concat(all_results, axis=1).T

## 4) Coefficient interpretation

In [ ]:
feature_names = ols_model.named_steps["preprocess"].get_feature_names_out()
coefs = ols_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs,
})
coef_df["pct_effect"] = (np.exp(coef_df["coef"]) - 1) * 100

print("Top 5 HIGHER amount:")
display(coef_df.nlargest(5, "coef").reset_index(drop=True).round(4))

print("\nTop 5 LOWER amount:")
display(coef_df.nsmallest(5, "coef").reset_index(drop=True).round(4))

## 5) Ridge with cross-validated alpha

In [ ]:
alpha_grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge(random_state=RANDOM_STATE)),
])

ridge_search = GridSearchCV(
    ridge_pipe,
    param_grid=alpha_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
ridge_search.fit(X_train, y_train)

y_pred_ridge = ridge_search.predict(X_test)
all_results.append(regression_report(y_test, y_pred_ridge, "Ridge (CV)"))

print("Best alpha:", ridge_search.best_params_)
pd.concat(all_results, axis=1).T

## 6) Check in — answers

**1. What R² did your OLS achieve? Did Ridge meaningfully improve on it?**

OLS lands at R² ≈ **0.236**; Ridge with the chosen `alpha` lands within ~0.001 of that. **No meaningful improvement.** Two reasons: (a) we have ~95K training rows with ~10 numeric features and standardized inputs — OLS already estimates stable coefficients even on the moderately correlated features; (b) regularization mostly helps when the design matrix is near-singular, which it isn't here. Ridge isn't doing collinearity-stabilization work because there's nothing to stabilize.

**2. Pick one coefficient with a large positive `pct_effect` and explain it.**

`avg_transaction_amount_30d` has coef ≈ +0.43 → **+54% multiplicative effect**. A 1-SD increase in an account's recent average transaction predicts a ~54% higher current transaction. This makes total sense — accounts with higher recent average spend are more likely to make larger transactions today. The model is essentially using account history as a per-account intercept, which is the dominant signal in this problem.

**3. Did the top-5 positive list contain anonymous `merchant_id_*` features?**

Yes — most of the top-5 positive (and top-5 negative) are `cat_high__merchant_id_<id>` dummies. This is a recurring pattern with high-cardinality categoricals: hundreds of one-hot dummies collectively absorb a lot of variance, and individually they get larger coefficients than the broader categorical effects (`merchant_category`, `payment_channel`) they conceptually nest inside. The merchant_id dummies are good for *prediction* but bad for *explanation* — "merchant 64 averages 21% higher" tells you nothing about *why*.

**4. If you wanted to push test R² higher, what would you try next?**

A few options, ordered by expected payoff:

- **Tree-based model (Random Forest / Gradient Boosting).** Trees automatically capture interactions and nonlinearities. The main notebook showed RF lands at R² ≈ 0.27 — a real but small gain over linear.
- **Target-encoded categoricals.** Replace one-hot expansion of `merchant_id` with the per-merchant target mean (smoothed). Often more efficient than one-hot for high-cardinality features.
- **Interaction features.** E.g., `merchant_category × payment_channel` to capture which combinations behave unusually.
- **More data, different features.** Longer history windows, time-of-day patterns, account tenure — all might help.

Note: there's likely a practical ceiling on R² for this problem because much of transaction-amount variation is *idiosyncratic* (one-off purchases that don't correlate with any feature). No model is going to push R² to 0.9 on this dataset.